# Freemarket Medallion Pipeline — orchestrator

This notebook **orchestrates and narrates**; all transformation logic lives in the tested
`src/` package (see `plan/SPEC.md` § Engineering Standards). Run top-to-bottom
(Restart & Run All) from an empty warehouse.

Layers so far: **Bronze** (`raw` monthly → `live` consolidated + counterparty/fees) →
**Silver `core`** (companies + groups unpick, FX rate attached to transactions & fees).

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config, warehouse, bronze, silver_core
from src.pipeline import build_foundation
config.WAREHOUSE_PATH

PosixPath('/Users/hallabehery/Documents/My Projects/FM-Data-Engineer-Test-HS/submission/warehouse.duckdb')

## 1. Foundation — six schemas

In [2]:
con = warehouse.connect()
build_foundation(con)

22:03:31 | INFO  | [foundation] schemas ready: raw, live, core, shape, data_mart, curated


('raw', 'live', 'core', 'shape', 'data_mart', 'curated')

## 2. Bronze `raw` — transactional sheets split per month

In [3]:
for report in bronze.build_raw_monthly(con):
    print(report)

22:03:31 | INFO  | [bronze.raw.deposits] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


22:03:31 | INFO  | [bronze.raw.withdrawals] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


[bronze.raw.deposits] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[bronze.raw.withdrawals] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


## 3. Bronze `live` — consolidate + land counterparty & fees

In [4]:
for report in bronze.build_live(con):
    print(report)

22:03:31 | INFO  | [bronze.live.deposits] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


22:03:31 | INFO  | [bronze.live.withdrawals] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


22:03:31 | INFO  | [bronze.live.counterparty] in=1585 out=1585 kept=1585 quarantined=0 dropped=0 conserved=yes


22:03:31 | INFO  | [bronze.live.fees] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


[bronze.live.deposits] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[bronze.live.withdrawals] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes
[bronze.live.counterparty] in=1585 out=1585 kept=1585 quarantined=0 dropped=0 conserved=yes
[bronze.live.fees] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


## 4. Silver `core` — companies unpick

In [5]:
print(silver_core.build_companies(con))

22:03:31 | INFO  | [silver.core.companies] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes


[silver.core.companies] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes


## 5. Silver `core` — groups unpick

In [6]:
print(silver_core.build_groups(con))

22:03:31 | INFO  | [silver.core.groups] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


[silver.core.groups] in=13 out=13 kept=13 quarantined=0 dropped=0 conserved=yes


## 6. Silver `core` — FX rate attached to transactions & fees
Attach the as-of FX rate to every deposit, withdrawal and fee using the tested FX unit
(`src/fx.py`). The settlement instant is `Tx Date + Tx Time` for transactions and the
fee `Date` (midnight) for fees. Rates are **attached, not applied** — GBP is computed in
Silver `shape`. Rows with no rate carry a `fx_quarantine_reason` and are **kept** here.

In [7]:
for report in silver_core.build_fx_attached(con):
    print(report)

22:03:32 | INFO  | [silver.core.deposits_fx] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes


22:03:32 | INFO  | [silver.core.withdrawals_fx] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes


22:03:32 | INFO  | [silver.core.fees_fx] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


22:03:32 | INFO  | [silver.core.fees_fx] flagged for FX quarantine (kept, not dropped): 42


[silver.core.deposits_fx] in=6383 out=6383 kept=6383 quarantined=0 dropped=0 conserved=yes
[silver.core.withdrawals_fx] in=6599 out=6599 kept=6599 quarantined=0 dropped=0 conserved=yes
[silver.core.fees_fx] in=21921 out=21921 kept=21921 quarantined=0 dropped=0 conserved=yes


In [8]:
# Sample: attached rate per row (GBP -> 1.0)
con.sql('''
    SELECT "Transaction ID", "Tx Currency", "Tx Value (CCY)",
           fx_instant_ms, fx_rate, fx_quarantine_reason
    FROM core.deposits ORDER BY "Transaction ID" LIMIT 8
''').df()

,Transaction ID,Tx Currency,Tx Value (CCY),fx_instant_ms,fx_rate,fx_quarantine_reason
0,334619.0,USD,1.600000e+06,1751358839203,0.726732,<NA>
1,334687.0,USD,1.500000e+06,1751344231990,0.727605,<NA>
2,334706.0,USD,2.673270e+04,1751352011240,0.727875,<NA>
3,334719.0,EUR,1.783262e+05,1751353591173,0.856944,<NA>
4,334740.0,EUR,1.500000e+04,1751382726537,0.858816,<NA>
5,334771.0,EUR,5.396000e+03,1751360459790,0.857017,<NA>
6,334776.0,USD,2.000000e+05,1751368192917,0.725849,<NA>
7,334785.0,SEK,3.790666e+06,1751361805127,0.076937,<NA>


In [9]:
# Data-quality: any rows flagged for FX quarantine (kept, not dropped), by reason
con.sql('''
    SELECT 'deposits' AS tbl, fx_quarantine_reason, COUNT(*) n FROM core.deposits WHERE fx_quarantine_reason IS NOT NULL GROUP BY 1,2
    UNION ALL SELECT 'withdrawals', fx_quarantine_reason, COUNT(*) FROM core.withdrawals WHERE fx_quarantine_reason IS NOT NULL GROUP BY 1,2
    UNION ALL SELECT 'fees', fx_quarantine_reason, COUNT(*) FROM core.fees WHERE fx_quarantine_reason IS NOT NULL GROUP BY 1,2
    ORDER BY 1,2
''').df()

,tbl,fx_quarantine_reason,n
0,fees,fx_missing_currency,42


## Inspect the built warehouse
Reopen `submission/warehouse.duckdb` **read-only** to see all tables built so far.

In [10]:
con.close()
import duckdb
inspect = duckdb.connect(str(config.WAREHOUSE_PATH), read_only=True)
tables = inspect.sql('''
    SELECT table_schema, COUNT(*) AS n_tables
    FROM information_schema.tables GROUP BY table_schema ORDER BY table_schema
''').df()
inspect.close()
tables

,table_schema,n_tables
0,core,5
1,live,4
2,raw,12
